# 第72章 交互箱线图（px.box）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 9 / 18 步：交互观察分布与矩阵**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互直方图（px.histogram）  →  **本章任务：** 交互箱线图（px.box）  →  **下一步：** 交互小提琴图（px.violin）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

商品订单里一堆客单价数字排成几列，光看平均值很容易被几个高额订单带偏；箱线图把每组的中位数、上四分位数、下四分位数和异常点一次呈现在眼前，而 Plotly 画出的交互箱线图还能把鼠标悬停在点上直接读出具体数值。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互箱线图（px.box）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互箱线图（px.box）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互箱线图（px.box）」并读出其中的结论。


## 适用场景

**背景引入**：商品订单里一堆客单价数字排成几列，光看平均值很容易被几个高额订单带偏；箱线图把每组的中位数、上四分位数、下四分位数和异常点一次呈现在眼前，而 Plotly 画出的交互箱线图还能把鼠标悬停在点上直接读出具体数值。要快速回答“哪一类商品整体更贵、更分散、有没有明显异常的订单”这类问题，它比翻阅一堆表格直观得多。（好比给一组数据做“体检报告”：箱体中间的线是中位数，箱体上下沿是四分位数（中间一半人所在），伸出的须是大多数人波动的范围，离群的圆点是被拉高或拉低的几个异常值，一眼看出整体水平、波动和两极。）

比较类别组的中位数、离散程度和潜在异常。


## 数据结构

一列分类和一列连续数值。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 points="outliers" 改为 points="all" 或 points=False，观察显示点数量的差异
2. 添加 notched=True 参数，对比缺口箱线图与标准箱线图的中位数比较效果
3. 修改 hover_data 增加补充字段，说明悬浮信息对异常点溯源的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.box()`、`fig.update_layout()`、`fig.show()` | 比较类别组的中位数、离散程度和潜在异常。 | 点全部显示造成拥挤 |
| 进阶变体 | `px.box()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 把异常点自动判为错误 |
| 关键参数 | `points` | 显示点 | 点全部显示造成拥挤 |
| 关键参数 | `notched` | 缺口 | 把异常点自动判为错误 |
| 关键参数 | `color` | 分组 | 不同组样本量不可见 |
| 关键参数 | `hover_data` | 补充信息 | 点全部显示造成拥挤 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-72 -->
### 数学推导｜箱线图的四分位数与异常界限

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜用分位数切分排序数据。** $Q_1$、$Q_2$、$Q_3$ 分别对应累计比例 25%、50%、75%。

**第 2 步｜中间一半数据的跨度是**

$$
IQR=Q_3-Q_1
$$

**第 3 步｜把箱体向两侧延伸 1.5 个 IQR。** 下、上界分别为 $L=Q_1-1.5IQR$、$U=Q_3+1.5IQR$。箱线图的“须”通常落到界内最远的实际观测，而不是直接画到 $L$、$U$。

**把上面的关系收束为本章计算式：**

$$
IQR=Q_3-Q_1,\qquad [L,U]=[Q_1-1.5IQR,\ Q_3+1.5IQR]
$$

**符号解释：** $Q_1$、$Q_3$ 是第一和第三四分位数，IQR 描述中间 50% 数据的跨度。

**代码对应：** 用 `quantile([.25, .5, .75])` 复核图中的箱体和中位数。

**使用边界：** 落在界限外的是统计异常点，不等于错误数据，更不能自动删除。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.box(
    orders,
    x="category",
    y="order_value",
    color="category",
    points="outliers",
    title="品类客单价箱线图",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="客单价（元）", showlegend=False
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上一步画的是只显示异常点的默认箱线图。请把 `points="outliers"` 改成 `points="all"`，让所有订单点都显示出来，观察同一品类下的点是否拥挤、能否看出样本密度；再给 `px.box(...)` 加上 `notched=True`，对比缺口箱线图与普通箱线图在比较中位数时有什么不同。在下方填写代码并运行，然后对照答案自检。


In [ ]:
try:
    # 请在下方填写代码
    # 任务：上一步画的是默认箱线图。请在下面把 points 改为 "all"，
    #      并增加 notched=True，观察显示点的数量和缺口中位数比较效果。
    # 提示：基于 orders，x="category", y="order_value", color="category"。

    # TODO: 在 px.box(...) 里补全 points 和 notched 参数
    fig = px.box(
        orders,
        x="category",
        y="order_value",
        color="category",
        title="品类客单价箱线图（练一练）",
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.box(
    orders,
    x="region",
    y="order_value",
    color="channel",
    points="suspectedoutliers",
    notched=True,
    title="区域渠道客单价",
)
fig.update_layout(
    xaxis_title="区域", yaxis_title="客单价（元）", legend_title="渠道"
)
fig.show()


## 参数说明

- points：显示点
- notched：缺口
- color：分组
- hover_data：补充信息


## 结果解读

读取箱体、中位数和须；悬浮异常点确认其所属类别和字段。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 点全部显示造成拥挤
- 把异常点自动判为错误
- 不同组样本量不可见


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：按渠道分组箱线，比较渠道间的金额分布
    # 【目标】换一个 x 分组变量，比较不同渠道的分布差异。
    import plotly.express as px

    # 起点示例(已可运行)：x 换成 channel，看不同渠道的客单价分布。
    fig = px.box(
        orders,
        x="channel",
        y="order_value",
        color="channel",
        points="outliers",
        title="分渠道客单价箱线图",
    )
    fig.update_layout(
        xaxis_title="渠道", yaxis_title="客单价（元）", showlegend=False
    )
    fig.show()

    # ---- 反思记录：不同渠道的中位数与离群点有何不同 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互箱线图比较分布摘要，并通过Hover查看具体异常值。


### 你已经掌握

- 判断交互箱线图（px.box）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `points` | 显示点 |
| `notched` | 缺口 |
| `color` | 分组 |
| `hover_data` | 补充信息 |


### 需要注意

- 点全部显示造成拥挤
- 把异常点自动判为错误
- 不同组样本量不可见


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案
fig = px.box(
    orders,
    x="category",
    y="order_value",
    color="category",
    points="all",
    notched=True,
    title="品类客单价箱线图（显示全部点 + 缺口）",
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
fig = px.box(
    orders,
    x="category",
    y="items",
    color="region",
    points="all",
    title="品类购买件数分布",
)
fig.update_traces(jitter=0.25, pointpos=0)
fig.update_layout(
    xaxis_title="品类", yaxis_title="购买件数", legend_title="区域"
)
fig.show()
